# Project: Investigate movie data

## Reading and understanding the data

In [42]:
import pandas as pd

movies_raw = pd.read_csv("data/movies/movies.csv")
print(movies_raw.shape)
print(movies_raw.dtypes)

(3201, 16)
Title                         str
US Gross                  float64
Worldwide Gross           float64
US DVD Sales              float64
Production Budget         float64
Release Date                  str
MPAA Rating                   str
Running Time min          float64
Distributor                   str
Source                        str
Major Genre                   str
Creative Type                 str
Director                      str
Rotten Tomatoes Rating    float64
IMDB Rating               float64
IMDB Votes                float64
dtype: object


In [43]:
movies_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 3201 entries, 0 to 3200
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Title                   3200 non-null   str    
 1   US Gross                3194 non-null   float64
 2   Worldwide Gross         3194 non-null   float64
 3   US DVD Sales            564 non-null    float64
 4   Production Budget       3200 non-null   float64
 5   Release Date            3201 non-null   str    
 6   MPAA Rating             2596 non-null   str    
 7   Running Time min        1209 non-null   float64
 8   Distributor             2969 non-null   str    
 9   Source                  2836 non-null   str    
 10  Major Genre             2926 non-null   str    
 11  Creative Type           2755 non-null   str    
 12  Director                1870 non-null   str    
 13  Rotten Tomatoes Rating  2321 non-null   float64
 14  IMDB Rating             2988 non-null   float64
 15

The non-null count varies between the different columns. There is only one column that has a non-null value for each of the 3201 entries, Release Date. There are no columns that are completely empty. 

In [44]:
movies_raw.head(3)

,Title,US Gross,Worldwide Gross,US DVD Sales,Production Budget,Release Date,MPAA Rating,Running Time min,Distributor,Source,Major Genre,Creative Type,Director,Rotten Tomatoes Rating,IMDB Rating,IMDB Votes
0,The Land Girls,146083.0,146083.0,NaN,8000000.0,Jun 12 1998,R,NaN,Gramercy,NaN,NaN,NaN,NaN,NaN,6.1,1071.0
1,"First Love, Last Rites",10876.0,10876.0,NaN,300000.0,Aug 07 1998,R,NaN,Strand,NaN,Drama,NaN,NaN,NaN,6.9,207.0
2,I Married a Strange Person,203134.0,203134.0,NaN,250000.0,Aug 28 1998,NaN,NaN,Lionsgate,NaN,Comedy,NaN,NaN,NaN,6.8,865.0


Each row represents a movie, three examples can be seen above.

## Cleaning the data

In [45]:
movies = movies_raw.copy()

### Cleaning dates (converting to datetime)

In [46]:
missing_dates_before = movies["Release Date"].isna().sum()
print("Missing values before", missing_dates_before)

Missing values before 0


In [47]:
movies["Release Date"] = pd.to_datetime(movies["Release Date"], errors="coerce")

In [48]:
missing_dates_after = movies["Release Date"].isna().sum()
print("Missing values after", missing_dates_after)

Missing values after 0


In [49]:
print(movies.dtypes)
movies.head(3)

Title                                str
US Gross                         float64
Worldwide Gross                  float64
US DVD Sales                     float64
Production Budget                float64
Release Date              datetime64[us]
MPAA Rating                          str
Running Time min                 float64
Distributor                          str
Source                               str
Major Genre                          str
Creative Type                        str
Director                             str
Rotten Tomatoes Rating           float64
IMDB Rating                      float64
IMDB Votes                       float64
dtype: object


,Title,US Gross,Worldwide Gross,US DVD Sales,Production Budget,Release Date,MPAA Rating,Running Time min,Distributor,Source,Major Genre,Creative Type,Director,Rotten Tomatoes Rating,IMDB Rating,IMDB Votes
0,The Land Girls,146083.0,146083.0,NaN,8000000.0,1998-06-12,R,NaN,Gramercy,NaN,NaN,NaN,NaN,NaN,6.1,1071.0
1,"First Love, Last Rites",10876.0,10876.0,NaN,300000.0,1998-08-07,R,NaN,Strand,NaN,Drama,NaN,NaN,NaN,6.9,207.0
2,I Married a Strange Person,203134.0,203134.0,NaN,250000.0,1998-08-28,NaN,NaN,Lionsgate,NaN,Comedy,NaN,NaN,NaN,6.8,865.0


In [50]:
movies["Release Date"].describe()

count                          3201
mean     1998-12-06 23:41:06.354264
min             1928-12-31 00:00:00
25%             1996-09-20 00:00:00
50%             2001-12-14 00:00:00
75%             2005-12-16 00:00:00
max             2046-12-31 00:00:00
Name: Release Date, dtype: object

The release dates in the future will be handled in the "Values that cannot be right" section.

### Duplicates

Converting all the movie titles temporarily into lowercase to check if there are duplicates with different upper/lowercase variants:

In [51]:
movies_lowercase_title = movies.copy()
movies_lowercase_title["Title"] = movies_lowercase_title["Title"].str.lower()
movies_lowercase_title.head(3)

,Title,US Gross,Worldwide Gross,US DVD Sales,Production Budget,Release Date,MPAA Rating,Running Time min,Distributor,Source,Major Genre,Creative Type,Director,Rotten Tomatoes Rating,IMDB Rating,IMDB Votes
0,the land girls,146083.0,146083.0,NaN,8000000.0,1998-06-12,R,NaN,Gramercy,NaN,NaN,NaN,NaN,NaN,6.1,1071.0
1,"first love, last rites",10876.0,10876.0,NaN,300000.0,1998-08-07,R,NaN,Strand,NaN,Drama,NaN,NaN,NaN,6.9,207.0
2,i married a strange person,203134.0,203134.0,NaN,250000.0,1998-08-28,NaN,NaN,Lionsgate,NaN,Comedy,NaN,NaN,NaN,6.8,865.0


In [52]:
duplicates = movies_lowercase_title[movies_lowercase_title.duplicated(subset=["Title"])]
all_duplicates = movies_lowercase_title[movies_lowercase_title["Title"].isin(duplicates["Title"])].sort_values("Title")
all_duplicates

,Title,US Gross,Worldwide Gross,US DVD Sales,Production Budget,Release Date,MPAA Rating,Running Time min,Distributor,Source,Major Genre,Creative Type,Director,Rotten Tomatoes Rating,IMDB Rating,IMDB Votes
25,"20,000 leagues under the sea",28200000.0,2.820000e+07,NaN,5000000.0,1954-12-23,NaN,NaN,Walt Disney Pictures,Based on Book/Short Story,Adventure,NaN,Richard Fleischer,92.0,NaN,NaN
26,"20,000 leagues under the sea",8000000.0,8.000000e+06,NaN,200000.0,2016-12-24,NaN,NaN,NaN,Based on Book/Short Story,Adventure,NaN,NaN,NaN,NaN,NaN
2406,a nightmare on elm street,63075011.0,1.051750e+08,NaN,35000000.0,2010-04-30,R,NaN,Warner Bros.,NaN,Horror,NaN,NaN,13.0,5.3,12554.0
654,a nightmare on elm street,25504513.0,2.550451e+07,NaN,1800000.0,1984-11-09,NaN,NaN,New Line,Original Screenplay,Horror,Fantasy,Wes Craven,95.0,5.3,12554.0
48,alice in wonderland,0.0,0.000000e+00,NaN,3000000.0,1951-07-28,NaN,NaN,RKO Radio Pictures,Based on Book/Short Story,Musical,NaN,NaN,20.0,6.7,63458.0
1138,alice in wonderland,334191110.0,1.023291e+09,70909558.0,200000000.0,2010-03-05,PG,NaN,Walt Disney Pictures,Based on Book/Short Story,Adventure,Fantasy,Tim Burton,51.0,6.7,63458.0
67,around the world in 80 days,42000000.0,4.200000e+07,NaN,6000000.0,1956-10-17,PG,NaN,United Artists,Based on Book/Short Story,Adventure,NaN,NaN,73.0,5.6,21516.0
1238,around the world in 80 days,24004159.0,7.200416e+07,NaN,110000000.0,2004-06-16,PG,120.0,Walt Disney Pictures,Remake,Adventure,Historical Fiction,Frank Coraci,30.0,5.6,21516.0
85,ben-hur,9000000.0,9.000000e+06,NaN,3900000.0,2025-12-30,NaN,NaN,MGM,Based on Book/Short Story,Adventure,NaN,NaN,NaN,8.2,58510.0
86,ben-hur,73000000.0,7.300000e+07,NaN,15000000.0,1959-11-18,NaN,NaN,MGM,Based on Book/Short Story,Adventure,NaN,William Wyler,91.0,8.2,58510.0


Comparing the entries with duplicate titles shows that while two films may have the same title, they are not necessarily the same movie. For instance, the release date, gross, budget, and more are different.   

Are there any movies with the same title and the same release date? Those might be true duplicates.

In [53]:
movies_lowercase_title[movies_lowercase_title.duplicated(subset=["Title", "Release Date"], keep=False)]

,Title,US Gross,Worldwide Gross,US DVD Sales,Production Budget,Release Date,MPAA Rating,Running Time min,Distributor,Source,Major Genre,Creative Type,Director,Rotten Tomatoes Rating,IMDB Rating,IMDB Votes


There are no movies with the same title and the same release date.

### Checking for negative or unusual min/max numerical values

In [54]:
movies.dtypes

Title                                str
US Gross                         float64
Worldwide Gross                  float64
US DVD Sales                     float64
Production Budget                float64
Release Date              datetime64[us]
MPAA Rating                          str
Running Time min                 float64
Distributor                          str
Source                               str
Major Genre                          str
Creative Type                        str
Director                             str
Rotten Tomatoes Rating           float64
IMDB Rating                      float64
IMDB Votes                       float64
dtype: object

In [55]:
numerical_columns = ["US Gross", "Worldwide Gross", "US DVD Sales", "Production Budget", "Running Time min", 
                     "Rotten Tomatoes Rating", "IMDB Rating", "IMDB Votes"]

for column in numerical_columns:
    print(f"The min value for {column} is {movies[column].min()}")
    print(f"The max value for {column} is {movies[column].max()}")

The min value for US Gross is 0.0
The max value for US Gross is 760167650.0
The min value for Worldwide Gross is 0.0
The max value for Worldwide Gross is 2767891499.0
The min value for US DVD Sales is 618454.0
The max value for US DVD Sales is 352582053.0
The min value for Production Budget is 218.0
The max value for Production Budget is 300000000.0
The min value for Running Time min is 46.0
The max value for Running Time min is 222.0
The min value for Rotten Tomatoes Rating is 1.0
The max value for Rotten Tomatoes Rating is 100.0
The min value for IMDB Rating is 1.4
The max value for IMDB Rating is 9.2
The min value for IMDB Votes is 18.0
The max value for IMDB Votes is 519541.0


There are no unusual values for the numerical columns.

### Fixing titles with accents

Looking at the data, there are some movie titles where a letter with an accent has been replaced by some other character. These movie titles are usually in French or Czech, languages that uses accents frequently.

Here are the patterns I found:  
`È` should be `é`  
`‡` should be `à`  
`Í` should be `ê`  
`·` should be `á`  

In [56]:
# All affected rows now
characters = ["È", "‡", "Í", "·"]
pattern = "|".join(characters)

before = movies[movies["Title"].str.contains(pattern)]
before

,Title,US Gross,Worldwide Gross,US DVD Sales,Production Budget,Release Date,MPAA Rating,Running Time min,Distributor,Source,Major Genre,Creative Type,Director,Rotten Tomatoes Rating,IMDB Rating,IMDB Votes
40,AstÈrix aux Jeux Olympiques,999811.0,132999811.0,NaN,113500000.0,2008-07-04,Not Rated,NaN,Alliance,Based on Comic/Graphic Novel,Adventure,Fantasy,NaN,NaN,4.9,5620.0
113,De battre mon coeur s'est arrÍtÈ,1023424.0,8589831.0,NaN,5500000.0,2005-07-01,Not Rated,NaN,WellSpring,Original Screenplay,Drama,Contemporary Fiction,NaN,NaN,7.3,7295.0
137,Les BronzÈs 3: amis pour la vie,0.0,83833602.0,NaN,42000000.0,2006-02-01,NaN,NaN,NaN,NaN,Comedy,Contemporary Fiction,NaN,NaN,4.4,1254.0
729,LÈon,19284974.0,45284974.0,NaN,16000000.0,1994-11-18,R,NaN,Sony Pictures,Original Screenplay,Thriller/Suspense,Contemporary Fiction,Luc Besson,NaN,8.6,199762.0
1163,Le Fabuleux destin d'AmÈlie Poulain,33201661.0,174201661.0,NaN,10350000.0,2001-11-02,R,122.0,Miramax,Original Screenplay,Comedy,Contemporary Fiction,Jean-Pierre Jeunet,NaN,8.5,181085.0
1168,AimÈe & Jaguar,927107.0,927107.0,NaN,6800000.0,2000-08-11,Not Rated,NaN,Zeitgeist,Based on Book/Short Story,Drama,Dramatization,NaN,NaN,7.2,2974.0
1409,CachÈ,3647381.0,17147381.0,NaN,8000000.0,2005-12-23,R,NaN,Sony Pictures Classics,Original Screenplay,Drama,Contemporary Fiction,NaN,NaN,5.5,26.0
1573,DÈj‡ Vu,64038616.0,181038616.0,40502497.0,80000000.0,2006-11-22,PG-13,126.0,Walt Disney Pictures,Original Screenplay,Thriller/Suspense,Science Fiction,Tony Scott,NaN,7.0,66106.0
2049,Obsluhoval jsem anglickÈho kr·le,617228.0,7174984.0,NaN,4900000.0,2008-08-29,R,NaN,Sony Pictures Classics,Based on Book/Short Story,Black Comedy,Historical Fiction,NaN,NaN,7.4,3048.0


In [57]:
conversion_table = str.maketrans({"È": "é", "‡": "à", "Í": "ê", "·": "á"})
movies["Title"] = movies["Title"].str.translate(conversion_table)

In [58]:
# Do any rows match the pattern after?
movies[movies["Title"].str.contains(pattern)]

,Title,US Gross,Worldwide Gross,US DVD Sales,Production Budget,Release Date,MPAA Rating,Running Time min,Distributor,Source,Major Genre,Creative Type,Director,Rotten Tomatoes Rating,IMDB Rating,IMDB Votes


In [59]:
# What the changed rows look like now, identified by release date 
# (some extra movies shown that happen to have the same relase date)
# The important point is that the accented characters in the titles are fixed.

movies[movies["Release Date"].isin(before["Release Date"])]

,Title,US Gross,Worldwide Gross,US DVD Sales,Production Budget,Release Date,MPAA Rating,Running Time min,Distributor,Source,Major Genre,Creative Type,Director,Rotten Tomatoes Rating,IMDB Rating,IMDB Votes
40,Astérix aux Jeux Olympiques,999811.0,132999811.0,NaN,113500000.0,2008-07-04,Not Rated,NaN,Alliance,Based on Comic/Graphic Novel,Adventure,Fantasy,NaN,NaN,4.9,5620.0
113,De battre mon coeur s'est arrêté,1023424.0,8589831.0,NaN,5500000.0,2005-07-01,Not Rated,NaN,WellSpring,Original Screenplay,Drama,Contemporary Fiction,NaN,NaN,7.3,7295.0
137,Les Bronzés 3: amis pour la vie,0.0,83833602.0,NaN,42000000.0,2006-02-01,NaN,NaN,NaN,NaN,Comedy,Contemporary Fiction,NaN,NaN,4.4,1254.0
729,Léon,19284974.0,45284974.0,NaN,16000000.0,1994-11-18,R,NaN,Sony Pictures,Original Screenplay,Thriller/Suspense,Contemporary Fiction,Luc Besson,NaN,8.6,199762.0
909,Star Trek: Generations,75671262.0,120000000.0,NaN,38000000.0,1994-11-18,PG,NaN,Paramount Pictures,Based on TV,Adventure,Science Fiction,NaN,NaN,6.5,26465.0
1008,Everything Put Together,0.0,0.0,NaN,500000.0,2001-11-02,R,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.7,418.0
1163,Le Fabuleux destin d'Amélie Poulain,33201661.0,174201661.0,NaN,10350000.0,2001-11-02,R,122.0,Miramax,Original Screenplay,Comedy,Contemporary Fiction,Jean-Pierre Jeunet,NaN,8.5,181085.0
1168,Aimée & Jaguar,927107.0,927107.0,NaN,6800000.0,2000-08-11,Not Rated,NaN,Zeitgeist,Based on Book/Short Story,Drama,Dramatization,NaN,NaN,7.2,2974.0
1224,Autumn in New York,37752931.0,90717684.0,NaN,40000000.0,2000-08-11,PG-13,106.0,MGM,Original Screenplay,Drama,Contemporary Fiction,Joan Chen,21.0,4.8,9309.0
1245,Babylon A.D.,22532572.0,70216497.0,16787309.0,45000000.0,2008-08-29,PG-13,100.0,20th Century Fox,Based on Book/Short Story,Action,Science Fiction,Mathieu Kassovitz,7.0,5.3,27189.0


### Values that cannot be right
As was discovered during the data cleaning, there are release dates that are in the future.

In [66]:
movies_with_future_release_dates = movies[movies["Release Date"] > pd.Timestamp.today()]

print(movies_with_future_release_dates.shape)
movies_with_future_release_dates[["Title", "Release Date"]]

(16, 17)


,Title,Release Date
9,Duel in the Sun,2046-12-31
16,Wilson,2044-08-01
33,42nd Street,2033-03-09
90,The Best Years of Our Lives,2046-11-21
174,"Charge of the Light Brigade, The",2036-10-20
221,Cat People,2042-11-16
337,Fantasia,2040-11-13
382,A Guy Named Joe,2043-12-24
400,Gone with the Wind,2039-12-15
412,How Green Was My Valley,2041-10-28


There are 16 movies with release dates in the future.   
One solution could be to fill these values with NaT (not a time values).  
Or you could look up the correct dates on IMDB given the title and all the other information in the table and fill them in.  
Or you could create a new boolean column that is `True` for future dates, for filtering out the future dates when needed.

I choose to create a boolean column called `is_future_date`, and filter out the future dates as needed.

In [68]:
movies["is_future_date"] = movies["Release Date"] > pd.Timestamp.today()

In [70]:
non_future_movies = movies[~movies["is_future_date"]]
non_future_movies["Release Date"].describe()

count                          3185
mean     1998-09-24 06:41:28.916797
min             1928-12-31 00:00:00
25%             1996-08-23 00:00:00
50%             2001-11-21 00:00:00
75%             2005-11-23 00:00:00
max             2025-12-30 00:00:00
Name: Release Date, dtype: object

## Missing values

In [14]:
missing_titles = movies[movies["Title"].isna()]
missing_titles

,Title,US Gross,Worldwide Gross,US DVD Sales,Production Budget,Release Date,MPAA Rating,Running Time min,Distributor,Source,Major Genre,Creative Type,Director,Rotten Tomatoes Rating,IMDB Rating,IMDB Votes
3053,NaN,26403.0,3080493.0,NaN,3700000.0,Nov 03 2006,Not Rated,85.0,IFC Films,Original Screenplay,Thriller/Suspense,Contemporary Fiction,NaN,39.0,6.6,11986.0
